# Fase 2b — Clusterização (continuação): DBSCAN e Hierárquico

## Retomando a pergunta de negócio

> **Existem perfis demográfico-sociais distintos entre os candidatos, nas Eleições 2026? Combinando idade, escolaridade e patrimônio declarado, os candidatos se agrupam em segmentos claramente diferentes — e, se sim, como esses segmentos se distribuem em termos de gênero, cor/raça e estado civil?**

Este notebook continua o `02_clusterizacao.ipynb`, reaproveitando **a mesma base, as mesmas variáveis-driver e a mesma preparação (Versão 2)** — só troca o algoritmo. Lá vimos K-Means e Bisecting K-Means, dois algoritmos "baseados em centróide": vocês escolhem (ou estimam) `k` de antemão, e cada cluster é definido pela distância até um centro.

Aqui entram duas famílias bem diferentes:

- **DBSCAN** (baseado em densidade): não pede `k` — define cluster como uma região "densa" de pontos, separada por regiões "vazias". Consegue achar clusters de formato arbitrário (não só "bolhas" convexas como o K-Means) e marca explicitamente pontos isolados como **ruído**, em vez de forçá-los para dentro do cluster mais próximo.
- **Clusterização Hierárquica (HCLUST/aglomerativa)**: não escolhe um único `k` — constrói uma **árvore inteira** de agrupamentos, do nível "cada candidato é seu próprio cluster" até "todo mundo é um cluster só". Vocês cortam essa árvore (o dendrograma) na altura que quiserem, o que dá muito mais flexibilidade pra explorar diferentes granularidades sem re-treinar nada.

Mesmas variáveis-driver de antes: `IDADE`, `ANOS_ESTUDO` (anos de estudo equivalentes) e `Total_Bens_Log`. Mesmas variáveis de perfil, pra descrever os clusters depois de prontos: `DS_GENERO`, `DS_COR_RACA`, `DS_ESTADO_CIVIL`.


## Configuração

Use os **mesmos valores** que você usou na Fase 0/1/2, para carregar o arquivo certo.


In [ ]:
CARGO = "GOVERNADOR"   # mesmo valor usado na Fase 0/1/2
UF = None               # mesmo valor usado na Fase 0/1/2
ANO_ELEICAO = 2026


## Preparando a base

Repetimos aqui a preparação da **Versão 2** do `02_clusterizacao.ipynb` — outliers de patrimônio removidos por IQR, escolaridade convertida em anos de estudo equivalentes, e as 3 variáveis-driver padronizadas com `MinMaxScaler` — para que este notebook rode de forma independente e os resultados continuem comparáveis com o K-Means / Bisecting K-Means já vistos.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

SEMENTE = 42
np.random.seed(SEMENTE)  # mesmo racional de reprodutibilidade defensiva do notebook anterior

nome_uf = UF if UF is not None else 'BRASIL'
nome_cargo = CARGO.replace(' ', '_')
arquivo = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}.csv"

df = pd.read_csv(arquivo)
print(f"Carregado: {arquivo}  ->  {df.shape[0]} candidatos, {df.shape[1]} colunas")
df.head()


In [ ]:
ANOS_ESTUDO_POR_GRAU = {
    'ANALFABETO': 0,
    'LÊ E ESCREVE': 1,
    'ENSINO FUNDAMENTAL INCOMPLETO': 4,
    'ENSINO FUNDAMENTAL COMPLETO': 8,
    'ENSINO MÉDIO INCOMPLETO': 9,
    'ENSINO MÉDIO COMPLETO': 11,
    'SUPERIOR INCOMPLETO': 13,
    'SUPERIOR COMPLETO': 16,
    # 'NÃO DIVULGÁVEL' fica de fora de propósito — mesma decisão da Versão 2 do notebook anterior
}

df['ANOS_ESTUDO'] = df['DS_GRAU_INSTRUCAO'].map(ANOS_ESTUDO_POR_GRAU)
print("Candidatos sem correspondência (NÃO DIVULGÁVEL ou similar):", df['ANOS_ESTUDO'].isna().sum())


In [ ]:
colunas_driver = ['IDADE', 'ANOS_ESTUDO', 'Total_Bens_Log']

Q1, Q3 = df['Total_Bens_Log'].quantile([0.25, 0.75])
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

antes = df.shape[0]
df_cluster = df[(df['Total_Bens_Log'] <= limite_superior) & (df['ANOS_ESTUDO'].notna())].copy()
print(f"Removidos {antes - df_cluster.shape[0]} candidatos "
      f"(patrimônio extremo acima de ~R$ {np.expm1(limite_superior):,.0f}, ou escolaridade não divulgada)")
print(f"Base para clusterização: {df_cluster.shape[0]} candidatos")

scaler = MinMaxScaler()
X = scaler.fit_transform(df_cluster[colunas_driver])

# DS_SIT_TOT_TURNO fica de fora — a eleição de 2026 ainda não ocorreu, então vem toda nula na base atual.
colunas_perfil = ['DS_GENERO', 'DS_COR_RACA', 'DS_ESTADO_CIVIL']
colunas_perfil = [c for c in colunas_perfil if c in df_cluster.columns]

df_cluster[colunas_driver].describe()


### Baseline: K-Means (k=4), para comparação

Não estamos re-escolhendo `k` aqui — isso já foi feito no `02_clusterizacao.ipynb` (Versão 2), cruzando cotovelo, silhouette, Davies-Bouldin e Calinski-Harabasz. Rodamos o K-Means de novo, rapidamente, só para ter uma referência própria neste notebook e comparar DBSCAN e Hierárquico contra ela mais adiante.


In [ ]:
k_final = 4  # mesmo k escolhido no 02_clusterizacao.ipynb (Versão 2) — mantém as comparações justas

kmeans = KMeans(n_clusters=k_final, random_state=SEMENTE, n_init=10)
df_cluster['cluster_kmeans'] = kmeans.fit_predict(X)
df_cluster['cluster_kmeans'] = df_cluster['cluster_kmeans'].map(lambda x: chr(65 + x))

sil_kmeans = silhouette_score(X, df_cluster['cluster_kmeans'])
print(f"K-Means (k={k_final}) — silhouette: {sil_kmeans:.3f}")
df_cluster['cluster_kmeans'].value_counts().sort_index()


### Radar: a mesma ferramenta do notebook anterior

Trazemos de volta a função `plot_radar` usada no `02_clusterizacao.ipynb`, para manter a mesma linguagem visual comparando os clusters nas 3 variáveis-driver (na escala padronizada 0–1, o que torna os eixos comparáveis entre si).

DBSCAN e o método hierárquico, ao contrário do K-Means, **não calculam um centróide** como parte do algoritmo — DBSCAN define clusters por conectividade de densidade, e a hierarquia por sucessivas junções. Para reaproveitar o radar mesmo assim, usamos a **média das variáveis-driver dentro de cada cluster** (na escala padronizada) como um "centróide de fato" — só para fins de visualização. No DBSCAN, o ruído fica de fora: não é um cluster, então não tem um perfil médio que faça sentido comparar.


In [ ]:
from math import pi

def plot_radar(ax, centros, labels_eixos, labels_series, cores, titulo):
    n_eixos = len(labels_eixos)
    angulos = [n / float(n_eixos) * 2 * pi for n in range(n_eixos)]
    angulos += angulos[:1]

    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(labels_eixos)
    ax.set_ylim(0, 1)

    for i, linha in enumerate(centros):
        valores = list(linha) + [linha[0]]
        cor = cores[labels_series[i]]
        ax.plot(angulos, valores, linewidth=2, label=labels_series[i], color=cor)
        ax.fill(angulos, valores, alpha=0.15, color=cor)

    ax.set_title(titulo, y=1.12)


def radar_clusters(centroides_norm, labels_clusters, cores, titulo_base):
    """Desenha o radar 'todos juntos' e, em seguida, 'um por cluster' — mesmo par de gráficos do notebook anterior."""
    k = len(labels_clusters)

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    plot_radar(ax, centroides_norm, colunas_driver, labels_clusters, cores, f'{titulo_base} — todos os clusters juntos')
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
    plt.show()

    ncols = min(k, 3)
    nrows = -(-k // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 5.5 * nrows), subplot_kw=dict(polar=True))
    axes = axes.flatten() if k > 1 else [axes]

    for i, cl in enumerate(labels_clusters):
        plot_radar(axes[i], [centroides_norm[i]], colunas_driver, [cl], cores, f'{titulo_base} — Cluster {cl}')

    for j in range(len(labels_clusters), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()


---
# Parte 1 — DBSCAN

**A ideia:** DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*) não parte de um número de clusters. Ele varre os pontos e classifica cada um em três papéis, usando dois parâmetros:

- **`eps`**: o raio de uma "vizinhança" ao redor de cada ponto.
- **`min_samples`**: quantos pontos (incluindo o próprio) precisam estar dentro dessa vizinhança para ela ser considerada "densa".

A partir daí:
- **Ponto núcleo (core point):** tem pelo menos `min_samples` vizinhos dentro do raio `eps`.
- **Ponto de borda (border point):** não é núcleo, mas está na vizinhança de algum ponto núcleo — entra no cluster desse vizinho.
- **Ruído (noise):** não é núcleo nem está perto de nenhum — fica de fora de qualquer cluster, com rótulo `-1`.

Clusters se formam encadeando pontos núcleo conectados entre si (e seus vizinhos de borda). Duas diferenças importantes em relação ao K-Means/Bisecting K-Means:

1. **Não é preciso escolher `k`** — o número de clusters é uma *consequência* de `eps` e `min_samples`, não um parâmetro de entrada.
2. **Outliers são marcados, não forçados para dentro de um cluster.** No K-Means todo ponto pertence a algum cluster, mesmo que fique longe do centróide; no DBSCAN um candidato isolado vira ruído explicitamente — potencialmente mais fiel à realidade, já que podem sobrar outliers "menores" mesmo depois do corte por IQR feito na preparação.

Como o DBSCAN também é baseado em distância (Euclidiana, por padrão), usamos a mesma base padronizada (`X`, `MinMaxScaler`) da Versão 2 do notebook anterior.


### 1.1) Escolhendo eps: gráfico k-distância

Não existe fórmula fechada para `eps`. A heurística padrão (do artigo original do DBSCAN, Ester et al. 1996) é: para cada ponto, calcular a distância até seu `min_samples`-ésimo vizinho mais próximo, ordenar essas distâncias em ordem crescente, e procurar o "cotovelo" — o ponto em que a curva sobe abruptamente. Antes do cotovelo, os pontos estão em regiões densas (distância pequena e estável); depois dele, começam a aparecer pontos cada vez mais isolados.

Para `min_samples`, uma regra prática comum é usar pelo menos `dimensões + 1` (aqui, 3 variáveis-driver → mínimo 4); usamos `2 × dimensões` como ponto de partida.


In [ ]:
min_samples_teste = 2 * len(colunas_driver)  # heurística comum: min_samples >= dimensões + 1; usamos 2x as dimensões

vizinhos = NearestNeighbors(n_neighbors=min_samples_teste).fit(X)
distancias, _ = vizinhos.kneighbors(X)
k_distancias = np.sort(distancias[:, -1])  # distância de cada ponto ao seu min_samples-ésimo vizinho, ordenada crescente

# Sugestão automática do "cotovelo": o ponto mais distante da reta que liga o primeiro e o
# último ponto da curva (heurística geométrica simplificada, tipo "kneedle"). Sempre confira
# visualmente no gráfico antes de aceitar — é só um ponto de partida, não a resposta final.
n = len(k_distancias)
p1 = np.array([0, k_distancias[0]])
p2 = np.array([n - 1, k_distancias[-1]])
reta_norm = (p2 - p1) / np.linalg.norm(p2 - p1)
pontos = np.column_stack([np.arange(n), k_distancias]) - p1
projecao = np.outer(pontos @ reta_norm, reta_norm)
distancia_perp = np.linalg.norm(pontos - projecao, axis=1)
indice_cotovelo = int(np.argmax(distancia_perp))
eps_sugerido = k_distancias[indice_cotovelo]

plt.figure(figsize=(9, 5))
plt.plot(k_distancias, marker='.', markersize=3)
plt.axhline(eps_sugerido, color='tab:red', linestyle='--', label=f'eps sugerido ≈ {eps_sugerido:.3f}')
plt.axvline(indice_cotovelo, color='tab:red', linestyle=':', alpha=0.5)
plt.xlabel('Candidatos (ordenados pela distância)')
plt.ylabel(f'Distância ao {min_samples_teste}º vizinho mais próximo')
plt.title(f'Gráfico k-distância (min_samples={min_samples_teste}) — procure o "cotovelo"')
plt.legend()
plt.grid(True)
plt.show()

print(f"eps sugerido pela heurística do cotovelo: {eps_sugerido:.3f}")


### 1.2) Rodando o DBSCAN

Com `eps` e `min_samples` definidos, rodamos o DBSCAN e conferimos: quantos clusters ele encontrou, e que fração da base virou ruído. Se quase tudo virar ruído, `eps` está pequeno demais (ou `min_samples` grande demais); se tudo virar um cluster só, `eps` está grande demais.


In [ ]:
eps_dbscan = round(eps_sugerido, 2)  # ajuste conforme o gráfico acima, se quiser
min_samples_dbscan = min_samples_teste

dbscan = DBSCAN(eps=eps_dbscan, min_samples=min_samples_dbscan)
labels_dbscan_num = dbscan.fit_predict(X)

n_clusters_dbscan = len(set(labels_dbscan_num)) - (1 if -1 in labels_dbscan_num else 0)
n_ruido = int((labels_dbscan_num == -1).sum())

# Rótulos por ordem de tamanho (A = maior cluster, ...); ruído fica com rótulo próprio, sempre por último
tamanhos = pd.Series(labels_dbscan_num[labels_dbscan_num != -1]).value_counts()
mapa_letras = {cid: chr(65 + i) for i, cid in enumerate(tamanhos.index)}
df_cluster['cluster_dbscan'] = [mapa_letras.get(c, 'Ruído') for c in labels_dbscan_num]

ordem_dbscan = sorted(df_cluster['cluster_dbscan'].unique(), key=lambda c: (c == 'Ruído', c))
cores_dbscan = dict(zip([c for c in ordem_dbscan if c != 'Ruído'], sns.color_palette('Set2', n_colors=n_clusters_dbscan)))
cores_dbscan['Ruído'] = (0.6, 0.6, 0.6)

print(f"eps={eps_dbscan}, min_samples={min_samples_dbscan}")
print(f"Clusters encontrados: {n_clusters_dbscan}")
print(f"Pontos de ruído: {n_ruido} ({n_ruido / len(labels_dbscan_num):.1%})")

mask_nao_ruido = df_cluster['cluster_dbscan'] != 'Ruído'
if df_cluster.loc[mask_nao_ruido, 'cluster_dbscan'].nunique() >= 2:
    sil_dbscan = silhouette_score(X[mask_nao_ruido.values], df_cluster.loc[mask_nao_ruido, 'cluster_dbscan'])
    print(f"Silhouette (só pontos não-ruído): {sil_dbscan:.3f}")
else:
    sil_dbscan = float('nan')
    print("Menos de 2 clusters não-ruído — silhouette não se aplica. Ajuste eps/min_samples.")

df_cluster['cluster_dbscan'].value_counts().loc[ordem_dbscan]


### 1.3) Radar: perfil das variáveis-driver por cluster

O "centróide" de cada cluster aqui é a média das variáveis-driver padronizadas dentro dele (ver explicação acima) — o ruído fica de fora, já que não é um cluster.


In [ ]:
clusters_dbscan_reais = [c for c in ordem_dbscan if c != 'Ruído']
centroides_dbscan_norm = np.array([
    X[df_cluster['cluster_dbscan'].values == cl].mean(axis=0) for cl in clusters_dbscan_reais
])

radar_clusters(centroides_dbscan_norm, clusters_dbscan_reais, cores_dbscan, 'DBSCAN')


### 1.4) Visualizando os clusters (e o ruído)

Mesmos dois planos de fundo usados no K-Means (`IDADE`/`Total_Bens_Log` e `ANOS_ESTUDO`/`Total_Bens_Log`), agora com o ruído destacado como `x` cinza — repare que ele tende a se concentrar nas bordas da nuvem de pontos, não no meio.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (x_col, y_col) in zip(axes, [('IDADE', 'Total_Bens_Log'), ('ANOS_ESTUDO', 'Total_Bens_Log')]):
    for cl in ordem_dbscan:
        subset = df_cluster[df_cluster['cluster_dbscan'] == cl]
        marker = 'x' if cl == 'Ruído' else 'o'
        alpha = 0.6 if cl == 'Ruído' else 0.5
        ax.scatter(subset[x_col], subset[y_col], label=cl, color=cores_dbscan[cl], marker=marker, alpha=alpha, s=40)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{x_col} x {y_col}')

axes[0].legend(title='cluster', fontsize=8)
plt.suptitle(f'DBSCAN — eps={eps_dbscan}, min_samples={min_samples_dbscan} ({n_clusters_dbscan} clusters + ruído)', y=1.02)
plt.tight_layout()
plt.show()


### 1.5) Sensibilidade a eps e min_samples

DBSCAN é conhecido por ser sensível à escolha dos parâmetros — pequenas mudanças em `eps` podem juntar dois clusters em um só, ou espalhar ruído por toda a base. Em vez de confiar num único par de valores, testamos uma grade de combinações e olhamos como o número de clusters e o percentual de ruído reagem.


In [ ]:
faixa_eps = np.round(np.linspace(max(0.02, eps_dbscan * 0.4), eps_dbscan * 1.8, 8), 3)
faixa_min_samples = [4, 6, 8, 10]

resultados_grid = []
for eps_teste in faixa_eps:
    for ms_teste in faixa_min_samples:
        labels_teste = DBSCAN(eps=eps_teste, min_samples=ms_teste).fit_predict(X)
        n_cl = len(set(labels_teste)) - (1 if -1 in labels_teste else 0)
        pct_ruido = (labels_teste == -1).mean() * 100
        resultados_grid.append({'eps': eps_teste, 'min_samples': ms_teste, 'n_clusters': n_cl, 'pct_ruido': pct_ruido})

grid = pd.DataFrame(resultados_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(grid.pivot(index='min_samples', columns='eps', values='n_clusters'),
            annot=True, fmt='.0f', cmap='viridis', ax=axes[0])
axes[0].set_title('Número de clusters encontrados')

sns.heatmap(grid.pivot(index='min_samples', columns='eps', values='pct_ruido'),
            annot=True, fmt='.0f', cmap='Reds', ax=axes[1])
axes[1].set_title('% de pontos marcados como ruído')

plt.tight_layout()
plt.show()


> Se a coluna/linha escolhida (`eps_dbscan`, `min_samples_dbscan`) estiver numa região "instável" do mapa — onde vizinhos próximos do grid mudam bastante o número de clusters ou o % de ruído — vale a pena reconsiderar a escolha. Regiões "estáveis" (várias combinações vizinhas dando resultado parecido) são um sinal melhor de que o resultado reflete estrutura real dos dados, não um acidente de parâmetro.

### 1.6) Perfil demográfico-social dos clusters do DBSCAN

Mesma leitura da Parte 1 do notebook anterior: como gênero, cor/raça e estado civil se distribuem dentro de cada cluster de densidade — e, dessa vez, também dentro do grupo de ruído (candidatos que não se encaixaram em nenhuma região densa).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(colunas_perfil):
    tabela = pd.crosstab(df_cluster['cluster_dbscan'], df_cluster[col], normalize='index').mul(100)
    tabela = tabela.loc[ordem_dbscan]
    tabela.plot(kind='bar', stacked=True, ax=axes[i])
    axes[i].set_title(f'{col} por cluster DBSCAN (%)')
    axes[i].set_ylabel('%')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

resumo_dbscan = df_cluster.groupby('cluster_dbscan').agg(
    qtd_candidatos=('SQ_CANDIDATO', 'count'),
    idade_media=('IDADE', 'mean'),
    anos_estudo_medio=('ANOS_ESTUDO', 'mean'),
    patrimonio_mediano=('Total_Bens', 'median'),
).round(1)
resumo_dbscan.loc[ordem_dbscan]


### 1.7) Comparando com o K-Means

Duas leituras: a métrica de qualidade (silhouette, calculada só sobre os pontos não-ruído do DBSCAN, já que ruído por definição não pertence a nenhum cluster) e o tamanho dos grupos. Repare que a comparação de tamanhos já não é "clusters do mesmo total" — o DBSCAN deixou uma fatia de fora como ruído.


In [ ]:
print(f"Silhouette — K-Means (k={k_final}):                     {sil_kmeans:.3f}")
print(f"Silhouette — DBSCAN (só não-ruído, {n_clusters_dbscan} clusters):    {sil_dbscan:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df_cluster['cluster_kmeans'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='tab:gray')
axes[0].set_title(f'Tamanho dos clusters — K-Means (k={k_final})')

df_cluster['cluster_dbscan'].value_counts().loc[ordem_dbscan].plot(
    kind='bar', ax=axes[1], color=[cores_dbscan[c] for c in ordem_dbscan]
)
axes[1].set_title(f'Tamanho dos clusters — DBSCAN ({n_clusters_dbscan} clusters + ruído)')

plt.tight_layout()
plt.show()

pd.crosstab(df_cluster['cluster_kmeans'], df_cluster['cluster_dbscan'])


---
# Parte 2 — Clusterização Hierárquica (HCLUST)

**A ideia:** em vez de decidir `k` antes de rodar o algoritmo (K-Means) ou deixar `k` emergir de `eps`/`min_samples` (DBSCAN), a clusterização hierárquica aglomerativa constrói **a árvore inteira de agrupamentos possíveis**, de uma vez:

1. Começa com cada candidato como seu próprio cluster (`n` clusters).
2. A cada passo, junta os **dois clusters mais próximos** em um só.
3. Repete até sobrar um único cluster com todo mundo.

O resultado é o **dendrograma** — uma árvore em que a altura de cada junção mostra a distância entre os dois clusters unidos. Vocês escolhem `k` **depois** de ver a árvore inteira, cortando-a horizontalmente na altura que fizer mais sentido — sem precisar re-treinar nada para testar outro `k` (diferente do K-Means, em que cada `k` é um treino novo).

A peça que falta especificar é **como medir a distância entre dois clusters** (não só entre dois pontos) — é o que os diferentes critérios de *linkage* fazem.


### 2.1) Comparando critérios de linkage

- **`single`** (ligação simples): distância entre os dois pontos **mais próximos** dos dois clusters. Tende a formar clusters "em cadeia", alongados — sensível a pontos que servem de ponte entre grupos.
- **`complete`** (ligação completa): distância entre os dois pontos **mais distantes**. Tende a formar clusters compactos e de tamanho parecido, mas pode quebrar grupos legítimos que tenham alguns pontos afastados.
- **`average`** (ligação média): média de todas as distâncias par a par entre os dois clusters — um meio-termo entre `single` e `complete`.
- **`ward`**: minimiza o aumento de variância **dentro** dos clusters a cada junção — é o critério mais próximo, conceitualmente, do que o K-Means otimiza (WCSS), o que o torna o mais fácil de comparar com os resultados da Parte 1 do notebook anterior.

Note como `single` tende a produzir uma árvore "desbalanceada", com uma cadeia longa de junções tardias (efeito conhecido como *chaining*) — um sintoma visual de que esse critério não é uma boa escolha para esta base.


In [ ]:
metodos_linkage = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

linkages = {}
for ax, metodo in zip(axes, metodos_linkage):
    Z = linkage(X, method=metodo)
    linkages[metodo] = Z
    dendrogram(Z, no_labels=True, ax=ax, color_threshold=0)
    ax.set_title(f"linkage = '{metodo}'")
    ax.set_xlabel('candidatos')
    ax.set_ylabel('distância')

plt.tight_layout()
plt.show()


### 2.2) Cortando a árvore: escolhendo k

Seguimos com `ward`, pelo motivo explicado acima. A linha tracejada mostra onde cortamos o dendrograma para chegar a `k_hclust` clusters — mude `k_hclust` e rode a célula de novo para ver outras granularidades sem re-treinar nada; essa é a vantagem prática da hierarquia sobre o K-Means.


In [ ]:
metodo_escolhido = 'ward'  # minimiza variância dentro do cluster — mesmo critério que o K-Means otimiza, o que facilita comparar os dois
Z_ward = linkages[metodo_escolhido]

k_hclust = 4  # mesmo k do K-Means/DBSCAN, para comparação; ajuste conforme o dendrograma acima
limiar_corte = Z_ward[-(k_hclust - 1), 2]  # altura de junção logo abaixo da qual restam exatamente k_hclust clusters

plt.figure(figsize=(14, 6))
dendrogram(Z_ward, no_labels=True, color_threshold=limiar_corte)
plt.axhline(limiar_corte, color='gray', linestyle='--', label=f'corte para k={k_hclust}')
plt.title(f"Dendrograma (linkage='{metodo_escolhido}') — corte em k={k_hclust}")
plt.xlabel('candidatos')
plt.ylabel('distância')
plt.legend()
plt.show()

labels_hclust_num = fcluster(Z_ward, t=k_hclust, criterion='maxclust')

tamanhos_h = pd.Series(labels_hclust_num).value_counts()
mapa_letras_h = {cid: chr(65 + i) for i, cid in enumerate(tamanhos_h.index)}
df_cluster['cluster_hclust'] = [mapa_letras_h[c] for c in labels_hclust_num]

clusters_hclust_ordenados = sorted(df_cluster['cluster_hclust'].unique())
cores_hclust = dict(zip(clusters_hclust_ordenados, sns.color_palette('Set2', n_colors=k_hclust)))

sil_hclust = silhouette_score(X, df_cluster['cluster_hclust'])
print(f"Hierárquico ({metodo_escolhido}, k={k_hclust}) — silhouette: {sil_hclust:.3f}")
df_cluster['cluster_hclust'].value_counts().sort_index()


### 2.3) Radar: perfil das variáveis-driver por cluster

Mesma lógica da Parte 1: os "centróides" abaixo são a média de cada cluster nas variáveis-driver padronizadas — o método hierárquico também não calcula centróides como parte do algoritmo, mas reconstruímos um para fins de visualização, do mesmo jeito que fizemos para o DBSCAN.


In [ ]:
centroides_hclust_norm = np.array([
    X[df_cluster['cluster_hclust'].values == cl].mean(axis=0) for cl in clusters_hclust_ordenados
])

radar_clusters(centroides_hclust_norm, clusters_hclust_ordenados, cores_hclust, f'Hierárquico ({metodo_escolhido})')


### 2.4) Perfil demográfico-social dos clusters hierárquicos


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(colunas_perfil):
    pd.crosstab(df_cluster['cluster_hclust'], df_cluster[col], normalize='index').mul(100).plot(
        kind='bar', stacked=True, ax=axes[i]
    )
    axes[i].set_title(f'{col} por cluster hierárquico (%)')
    axes[i].set_ylabel('%')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

resumo_hclust = df_cluster.groupby('cluster_hclust').agg(
    qtd_candidatos=('SQ_CANDIDATO', 'count'),
    idade_media=('IDADE', 'mean'),
    anos_estudo_medio=('ANOS_ESTUDO', 'mean'),
    patrimonio_mediano=('Total_Bens', 'median'),
).round(1)
resumo_hclust


### 2.5) Comparando os três algoritmos

As tabelas de contingência abaixo cruzam os rótulos de cada par de algoritmos: se a maior parte dos candidatos de um cluster do K-Means cair concentrada em um único cluster do Hierárquico (em vez de espalhada), é sinal de que os dois algoritmos, apesar de construídos de formas bem diferentes, estão enxergando a mesma estrutura nos dados.


In [ ]:
comparacao = pd.DataFrame({
    'algoritmo': ['K-Means', 'DBSCAN', f'Hierárquico ({metodo_escolhido})'],
    'k_ou_clusters': [k_final, n_clusters_dbscan, k_hclust],
    'silhouette': [sil_kmeans, sil_dbscan, sil_hclust],
    'trata_ruido_separado': [False, True, False],
}).round(3)
display(comparacao)

print("K-Means x Hierárquico:")
display(pd.crosstab(df_cluster['cluster_kmeans'], df_cluster['cluster_hclust']))

print("DBSCAN x Hierárquico:")
display(pd.crosstab(df_cluster['cluster_dbscan'], df_cluster['cluster_hclust']))


---
## Fechamento: quatro algoritmos, quatro jeitos de ver os mesmos candidatos

Entre este notebook e o `02_clusterizacao.ipynb`, rodamos quatro algoritmos de clusterização na mesma pergunta de negócio e nas mesmas variáveis-driver:

| Algoritmo | Como decide `k` | Trata outliers | Forma dos clusters | Custo computacional |
|---|---|---|---|---|
| **K-Means** | escolhido antes (cotovelo/silhouette/DB/CH) | força todo ponto para o cluster mais próximo | esférica/convexa (assume) | baixo, escala bem |
| **Bisecting K-Means** | construído incrementalmente, 1→k | idem K-Means | idem K-Means | baixo, escala bem |
| **DBSCAN** | emerge de `eps`/`min_samples` | marca como ruído, separado | arbitrária (não-convexa) | médio; sensível a `eps`/`min_samples` |
| **Hierárquico (ward)** | escolhido depois, cortando o dendrograma | idem K-Means (todo ponto entra em algum cluster) | depende do linkage | alto — O(n²) ou pior; não escala para bases muito maiores |

Nenhum dos quatro é "o certo" — a escolha depende da pergunta: se vocês já sabem quantos perfis esperam encontrar e querem algo rápido e escalável, K-Means/Bisecting K-Means bastam. Se suspeitam que existem outliers genuínos que não deveriam ser forçados em nenhum grupo, DBSCAN é mais honesto. Se querem explorar várias granularidades sem re-treinar, ou entender a ordem em que os grupos se formam, a hierarquia dá essa visão que nenhum dos outros três oferece.

**Exercício sugerido:** repliquem aqui o mesmo kit de nomeação de clusters da Parte 1 do notebook anterior (ficha técnica + heatmap de desvio + rascunho automático), agora para `cluster_dbscan` e `cluster_hclust`.

Próxima etapa do pipeline: `03_regras_associacao.ipynb` (Apriori).
